## Scraping Respiratory Virus Detections Data

I will now scrape the respiratory virus detections data from the Canadian public health website. The goal is to extract 'Table 2' and convert it into a pandas DataFrame.

<a href="https://colab.research.google.com/github/techseeko/AAI614_Haidar/blob/main/Week-3/Saoud_VirusDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an exception for HTTP errors
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find 'Table 2'. Based on inspection, it might be the second table on the page.
    # More robust way: find the heading 'Table 2' and then find the next table sibling.
    table_heading = soup.find('h3', string=lambda text: text and 'Table 2' in text)
    if table_heading:
        table = table_heading.find_next_sibling('table')
    else:
        # Fallback if the heading isn't found, try to get the second table
        tables = soup.find_all('table')
        if len(tables) > 1:
            table = tables[1] # Assuming 'Table 2' is the second table
        else:
            table = None
            print("Could not find 'Table 2' on the page.")

    if table:
        headers = [th.text.strip() for th in table.find('thead').find_all('th')]
        data = []
        for row in table.find('tbody').find_all('tr'):
            cells = row.find_all('td')
            # Exclude rows that are just headers or empty
            if cells:
                data.append([cell.text.strip() for cell in cells])

        df = pd.DataFrame(data, columns=headers)
        print("DataFrame created successfully from Table 2.")
        display(df.head())
        display(df.info())
    else:
        print("No table found for extraction.")

except requests.exceptions.HTTPError as err:
    print(f"HTTP error occurred: {err}")
except requests.exceptions.RequestException as err:
    print(f"An error occurred: {err}")
except Exception as err:
    print(f"An unexpected error occurred: {err}")

An error occurred: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
